In [24]:
import tenseal as ts
import numpy as np

# Fix encryption parameters - need more coefficient modulus primes for multiplication depth
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192 * 4,
    coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 40, 40, 60]  # Added more levels
)
context.generate_galois_keys()
context.global_scale = 2**40                

# Test vector
c = [0, 1, 2, 3, 4]
c_enc = ts.ckks_vector(context, c)

def multiplication_test_encrypted(c_enc: ts.CKKSVector, i: int) -> ts.CKKSVector:
    """Perform i iterations of element-wise multiplication on encrypted vector"""
    result = c_enc
    for _ in range(i):
        result = result * result
    return result

def multiplication_test_normal(c: list, i: int) -> list:
    """Perform i iterations of element-wise multiplication on normal vector"""
    result = c.copy()
    for _ in range(i):
        result = [x * x for x in result]
    return result

# Test with 3 iterations
iterations = 3

print("Original vector:", c)
print()

# Normal vector multiplication
normal_result = multiplication_test_normal(c, iterations)
print(f"Normal vector result after {iterations} iterations:", normal_result)
print()

# Encrypted vector multiplication
encrypted_result = multiplication_test_encrypted(c_enc, iterations)
decrypted_result = encrypted_result.decrypt()
print(f"Encrypted vector result after {iterations} iterations:", decrypted_result)
print()

# Compare results
print("Results match:", np.allclose(normal_result, decrypted_result, rtol=1e-3))


Original vector: [0, 1, 2, 3, 4]

Normal vector result after 3 iterations: [0, 1, 256, 6561, 65536]

Encrypted vector result after 3 iterations: [4.3612828903860645e-09, 1.00001745749978, 256.0044945461923, 6561.114838446001, 65537.14658600862]

Results match: True
